Bringing over the same set up as my base model (which will be ran last for comparison)

In [19]:
import pickle
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


df = pd.read_pickle('../data/cleaned_games.pkl')

with open('../data/genre_columns.pkl', 'rb') as f:
    genre_features = pickle.load(f)

features = [
    'price',
    'year',
    'num_tags',
    'dev_success',
    'dev_had_success',
    'log_dev_experience',
    'num_devs',
] + list(genre_features)

X = df[features]
y = df['hit']

split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]



Now, with this as a basis, I will complete some model analysis before moving into my second layer, the main purpose of this project. I will start by separating th efeatures into subgroups to see if any one group is entirely carrying the model.

In [20]:
from matplotlib import cm


dev_features = [
    'dev_success',
    'dev_had_success',
    'log_dev_experience',
    'num_devs'
]
genre_features = list(genre_features)

def run_model(feature_list):
    X_train_sub = X_train[feature_list]
    X_test_sub = X_test[feature_list]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_sub)
    X_test_scaled = scaler.transform(X_test_sub)
    
    model = LogisticRegression(max_iter=3000, class_weight='balanced')
    model.fit(X_train_scaled, y_train)
    
    y_probs = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_probs > 0.5).astype(int)

    report = classification_report(y_test, y_pred, output_dict=True)
    f1 = report['1']['f1-score']
    precision = report['1']['precision']
    recall = report['1']['recall']
    
    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(cm, index=['Actual: Miss', 'Actual: Hit'], columns=['Predicted: Miss', 'Predicted: Hit'])
   
    # clean output
    result = {
        "F1": round(f1, 3),
        "Precision": round(precision, 3),
        "Recall": round(recall, 3),
        "Confusion Matrix": display(cm_df)
    }
    
    return result

In [21]:
run_model(dev_features)

,Predicted: Miss,Predicted: Hit
Actual: Miss,31342,1727
Actual: Hit,811,331


{'F1': 0.207, 'Precision': 0.161, 'Recall': 0.29, 'Confusion Matrix': None}

In [22]:
run_model(genre_features)

,Predicted: Miss,Predicted: Hit
Actual: Miss,18312,14757
Actual: Hit,416,726


{'F1': 0.087, 'Precision': 0.047, 'Recall': 0.636, 'Confusion Matrix': None}

In [23]:
run_model(features)

,Predicted: Miss,Predicted: Hit
Actual: Miss,32481,588
Actual: Hit,834,308


{'F1': 0.302, 'Precision': 0.344, 'Recall': 0.27, 'Confusion Matrix': None}

In [24]:
features_clean = [
    'price',
    'year',
    'dev_success',
    'dev_had_success',
    'log_dev_experience',
    'num_devs'
] + list(genre_features)

In [25]:
run_model(features_clean)

,Predicted: Miss,Predicted: Hit
Actual: Miss,31959,1110
Actual: Hit,875,267


{'F1': 0.212, 'Precision': 0.194, 'Recall': 0.234, 'Confusion Matrix': None}

In [26]:
results = {
      'Dev features only': run_model(dev_features),
      'Genre features only': run_model(genre_features),
      'All features (leaky)': run_model(features),
      'Clean features': run_model(features_clean),
}
summary = pd.DataFrame({k: {m: v[m] for m in ['F1','Precision','Recall']} for k, v in results.items()}).T
display(summary)

,Predicted: Miss,Predicted: Hit
Actual: Miss,31342,1727
Actual: Hit,811,331


,Predicted: Miss,Predicted: Hit
Actual: Miss,18312,14757
Actual: Hit,416,726


,Predicted: Miss,Predicted: Hit
Actual: Miss,32481,588
Actual: Hit,834,308


,Predicted: Miss,Predicted: Hit
Actual: Miss,31959,1110
Actual: Hit,875,267


,F1,Precision,Recall
Dev features only,0.207,0.161,0.290
Genre features only,0.087,0.047,0.636
All features (leaky),0.302,0.344,0.270
Clean features,0.212,0.194,0.234
